# Diniz Ch.4 — Supervised sinusoid denoising

Example 4.10：32 taps、sinusoid + −10 dB AWGN；比較 LMS、Sign-Error、Power-of-Two 與 NLMS。clean sinusoid 是 supervised training target。

來源：`Diniz-Chap4-Quantized-Error and Normalized LMS Algorithms.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters, data and factories

In [ ]:
n_samples, n_trials, seed, n_taps = 1000, 500, 410, 32
make_data = partial(data.denoising, n_samples, frequency=0.1, snr_db=-10)
models = {
  "LMS": partial(af.LMS, filter_order=n_taps - 1, mu=0.005 / 2),
  "Sign-Error LMS": partial(af.SignErrorLMS, filter_order=n_taps - 1, mu=0.001),
  "Power-of-Two LMS": partial(af.PowerOfTwoErrorLMS, filter_order=n_taps - 1, mu=0.001, bd=12),
  "NLMS": partial(af.NLMS, filter_order=n_taps - 1, mu=0.4, tau=1e-6),
}
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 3. Learning, time-domain reconstruction and spectra

In [ ]:
plot.learning(results, scale="log", window=21, title="Supervised sinusoid denoising")
plot.show()
result = results["NLMS"].representative
plot.lines({
  "Noisy input": result.x[-200:].real,
  "Clean target": result.d[-200:],
  "NLMS output": result.y[-200:].real
}, title="Tail reconstruction")
plot.show()
plot.spectrum({
  "Noisy input": result.x,
  "Clean target": result.d,
  **{
    name: res.representative.y
    for name, res in results.items()
  }
})
plot.show()
plot.frequency_response({name: np.conj(res.representative.w_final) for name, res in results.items()})
plot.show()
signal_power = np.mean(np.abs(result.d[result.valid])**2)
input_noise_power = np.mean((result.x[result.valid].real - result.d[result.valid].real)**2)
input_snr = stats.power_db(signal_power / input_noise_power)
for name, res in results.items():
  output_snr = stats.power_db(signal_power / stats.tail_mean(res.mse, tail=200))
  print(f"{name}: input SNR={input_snr:.3f} dB, tail output SNR={output_snr:.3f} dB")
training.summary(results)